# Exercises - The Carry Trade

#### Notation Commands

$$\newcommand{\Black}{\mathcal{B}}
\newcommand{\Blackcall}{\Black_{\mathrm{call}}}
\newcommand{\Blackput}{\Black_{\mathrm{put}}}
\newcommand{\EcondS}{\hat{S}_{\mathrm{conditional}}}
\newcommand{\Efwd}{\mathbb{E}^{T}}
\newcommand{\Ern}{\mathbb{E}^{\mathbb{Q}}}
\newcommand{\Tfwd}{T_{\mathrm{fwd}}}
\newcommand{\Tunder}{T_{\mathrm{bond}}}
\newcommand{\accint}{A}
\newcommand{\carry}{\widetilde{\cpn}}
\newcommand{\cashflow}{C}
\newcommand{\convert}{\phi}
\newcommand{\cpn}{c}
\newcommand{\ctd}{\mathrm{CTD}}
\newcommand{\disc}{Z}
\newcommand{\done}{d_{1}}
\newcommand{\dt}{\Delta t}
\newcommand{\dtwo}{d_{2}}
\newcommand{\flatvol}{\sigma_{\mathrm{flat}}}
\newcommand{\flatvolT}{\sigma_{\mathrm{flat},T}}
\newcommand{\float}{\mathrm{flt}}
\newcommand{\freq}{m}
\newcommand{\futprice}{\mathcal{F}(t,T)}
\newcommand{\futpriceDT}{\mathcal{F}(t+h,T)}
\newcommand{\futpriceT}{\mathcal{F}(T,T)}
\newcommand{\futrate}{\mathscr{f}}
\newcommand{\fwdprice}{F(t,T)}
\newcommand{\fwdpriceDT}{F(t+h,T)}
\newcommand{\fwdpriceT}{F(T,T)}
\newcommand{\fwdrate}{f}
\newcommand{\fwdvol}{\sigma_{\mathrm{fwd}}}
\newcommand{\fwdvolTi}{\sigma_{\mathrm{fwd},T_i}}
\newcommand{\grossbasis}{B}
\newcommand{\hedge}{\Delta}
\newcommand{\ivol}{\sigma_{\mathrm{imp}}}
\newcommand{\logprice}{p}
\newcommand{\logyield}{y}
\newcommand{\mat}{(n)}
\newcommand{\nargcond}{d_{1}}
\newcommand{\nargexer}{d_{2}}
\newcommand{\netbasis}{\tilde{\grossbasis}}
\newcommand{\normcdf}{\mathcal{N}}
\newcommand{\notional}{K}
\newcommand{\pfwd}{P_{\mathrm{fwd}}}
\newcommand{\pnl}{\Pi}
\newcommand{\price}{P}
\newcommand{\probexer}{\hat{\mathcal{P}}_{\mathrm{exercise}}}
\newcommand{\pvstrike}{K^*}
\newcommand{\refrate}{r^{\mathrm{ref}}}
\newcommand{\rrepo}{r^{\mathrm{repo}}}
\newcommand{\spotrate}{r}
\newcommand{\spread}{s}
\newcommand{\strike}{K}
\newcommand{\swap}{\mathrm{sw}}
\newcommand{\swaprate}{\cpn_{\swap}}
\newcommand{\tbond}{\mathrm{fix}}
\newcommand{\ttm}{\tau}
\newcommand{\value}{V}
\newcommand{\vega}{\nu}
\newcommand{\years}{\tau}
\newcommand{\yearsACT}{\tau_{\mathrm{act/360}}}
\newcommand{\yield}{Y}$$

Use the data set `famabliss_strips_2025-11-28.xlsx`.

It gives prices on **zero coupon bonds** with maturities of 1 through 5 years.
* These are prices per \$1 face value on bonds that only pay principal.
* Such bonds can be created from treasuries by *stripping* out their coupons.
* In essence, you can consider these prices as the discount factors $Z$, for maturity intervals 1 through 5 years.

In this problem, we focus on six dates: the month of **November** in 2020 through 2025.

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

In [7]:
df = pd.read_excel('../Data/famabliss_strips_2025-11-28.xlsx')
df['date'] = pd.to_datetime(df['date'])

# 2. Filter for November (month 11) between 2020 and 2025 inclusive
filtered_df = df[
    (df['date'].dt.month == 11) & 
    (df['date'].dt.year >= 2020) & 
    (df['date'].dt.year <= 2025)
]
print(filtered_df)

          date         1         2         3         4         5
821 2020-11-30  0.998839  0.997047  0.994464  0.988809  0.981515
833 2021-11-30  0.997523  0.988614  0.974796  0.958322  0.943772
845 2022-11-30  0.954375  0.918220  0.887608  0.857356  0.830593
857 2023-11-30  0.951080  0.911951  0.876957  0.842191  0.809378
869 2024-11-29  0.959187  0.920923  0.885440  0.850418  0.817795
881 2025-11-28  0.964936  0.932866  0.900919  0.868240  0.836057


# The Carry Trade

## 1.1

Suppose it is `November 2020`, and you determine to implement a carry trade with the following specification...

* Long `$100` million (market value, not face value) of the 5-year zero-coupon bond (maturing `Nov 2025`.)
* Short `$100` million (market value, not face value) of the 1-year zero-coupon bond (maturing `Nov 2021`.)
* Assume there is a `2%` haircut on each side of the trade, so it requires `$4` million of investor capital to initiate it.

1. Calculate the total profit and loss year-by-year.
1. Calculate the total return (`Nov 2025`) on the initial \\$4 million of investor capital.

#### Short position
* Each year you will roll over the short position to maintain a short `$100` million (market value) in the 1-year bond.
* This will require injecting more cash into the trade, as the expiring short will require more than `$100` million to close out. 
* In `Nov 2024`, no need to open a new short position, as your long position will (at that point) be a one-year bond.

#### Alternatives
The scheme above is for simplicity. You could try more interesting ways of setting the short position...
* Open a new short position sized to whatever is needed to cover the expiring short position.
* Set the short positions to duration-hedge the long position.

In [15]:
# Clean column names and convert to float
filtered_df.columns = [str(col) for col in filtered_df.columns]
for col in ['1', '2', '3', '4', '5']:
    filtered_df.loc[:, col] = pd.to_numeric(filtered_df[col], errors='coerce')

# Trade parameters 
notional = 100_000_000      # $100 million market value
initial_capital = 4_000_000 # $4 million
long_fv = notional / filtered_df.iloc[0]['5']  # Number of 5-yr bonds purchased

pnl_data = []
total_injections = 0

# Year-by-Year Calculation ---
years = len(filtered_df) - 1

for i in range(years):
    t_start = i
    t_end = i + 1
    
    # --- LONG POSITION ---
    years_left_end = 5 - (i + 1)
    p_long_end = 1.0 if years_left_end == 0 else filtered_df.iloc[t_end][str(years_left_end)]
    p_long_start = filtered_df.iloc[t_start][str(5 - i)]
    long_pnl_year = long_fv * (p_long_end - p_long_start)
    
    # --- SHORT POSITION ---
    # Skip the last short (Nov 2024 → Nov 2025)
    if i == years - 1:
        short_pnl_year = 0
        injection = 0
    else:
        p_short_start = filtered_df.iloc[t_start]['1']
        short_fv_at_maturity = notional / p_short_start
        short_pnl_year = notional - short_fv_at_maturity
        injection = max(0, short_fv_at_maturity - notional)
        total_injections += injection
    
    # Store results
    pnl_data.append({
        'Year': filtered_df.iloc[t_end]['date'].year,
        'Long P&L': long_pnl_year,
        'Short P&L': short_pnl_year,
        'Total P&L': long_pnl_year + short_pnl_year,
        'Injection': injection
    })


results_df = pd.DataFrame(pnl_data)
total_pnl = results_df['Total P&L'].sum()
total_capital_deployed = initial_capital + total_injections
total_return = total_pnl / initial_capital


print("Yearly Trade Performance:")
print(results_df[['Year', 'Total P&L', 'Injection']])

print(f"\nTotal P&L (2020-2025): ${total_pnl:,.2f}")
print(f"Total Capital (Initial + Injections): ${total_capital_deployed:,.2f}")
print(f"Total Return on Initial Capital: {total_return:.2%}")


Yearly Trade Performance:
   Year     Total P&L     Injection
0  2021 -2.479200e+06  1.162275e+05
1  2022 -7.453003e+06  2.483473e+05
2  2023 -2.300442e+06  4.780616e+06
3  2024 -3.310760e+05  5.143642e+06
4  2025  4.158168e+06  0.000000e+00

Total P&L (2020-2025): $-8,405,552.01
Total Capital (Initial + Injections): $14,288,833.17
Total Return on Initial Capital: -210.14%


## 1.2

How would this trade play out if the path of one-year spot rates equaled the forward rates observed in `2020`?

If the 1-year spot rates in future years exactly followed the forward rates implied by the 2020 yield curve, the carry trade would play out exactly as expected based on the initial curve. The long 5-year bond would accrue value consistently with the forward-implied discount factors, while the short 1-year bonds could be rolled each year at prices predicted by the forward rates. In this scenario, there would be no surprises from interest rate movements, so the profit and loss would be deterministic. If the 2020 curve was upward-sloping, the trade would likely generate positive carry, as the long bond earns more than the cost of rolling the short position; if the curve were downward-sloping, the opposite would occur.

## 1.3

Given Fact 3 of the *dynamic* (conditional) tests of the Expectations Hypothesis (EH), do you expect that as of `Nov 2025` the long-short trade above looks more or less favorable for `Nov 2025-2030` than it did for `Nov 2020-2025`?

Under Fact 3 of the dynamic (conditional) tests of the Expectations Hypothesis, forward rates tend to be biased predictors of future spot rates, often overestimating short-term rates when the term structure is upward-sloping. This means that by November 2025, the long-short carry trade looking ahead to 2025–2030 may appear less favorable than it did for 2020–2025. The long bond would likely offer lower expected incremental returns relative to the cost of shorting the 1-year bonds, because forward rates in 2025 may already incorporate much of the anticipated yield increases. In other words, the expected carry would be smaller, and the trade may generate lower profits compared to the earlier period.

***